## Import packages

In [1]:
%pip install --upgrade pip sagemaker boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 33.6 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
  Attempting uninstall: botocore
    Found existing installation: botocore 1.35.92
    Uninstalling botocore-1.35.92:
      Successfully uninstalled botocore-1.35.92
  Attempting uninstall: boto3
    Found existing installation: boto3 1.35.92
    Uninstalling boto3-1.35.92:
      Successfully uninstalled boto3-1.35.92
  Attempting uninstall: sagemaker
    Found existing installation: sagemaker 2.237.1
    Uninstalling sagemaker-2.237.1:
      Successfully uninstalled sagemaker-2.237.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mlflow sagemaker-mlflow

  Using cached sagemaker_mlflow-0.1.0-py3-none-any.whl.metadata (3.3 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import os
import json
import boto3
import numpy as np  
import pandas as pd 
import sagemaker
from time import gmtime, strftime, sleep

import zipfile
import warnings
warnings.filterwarnings('ignore')


/Users/bharathbeeravelly/.pyenv/versions/3.9.19/lib/python3.9/site-packages/pydantic/_internal/_fields.py:172: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[01/10/25 09:44:26] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=896828;file:///Users/bharathbeeravelly/.pyenv/versions/3.9.19/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=68373;file:///Users/bharathbeeravelly/.pyenv/versions/3.9.19/lib/python3.9/site-packages/botocore/credentials.py#1278\1278]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/bharathbeeravelly/Library/Application Support/sagemaker/config.yaml


In [4]:
print(f'Sagemaker version: {sagemaker.__version__}')
print(f'Boto3 version: {boto3.__version__}')

Sagemaker version: 2.237.3
Boto3 version: 1.35.96


In [ ]:
# Create a new boto3 session
boto_session = boto3.Session()

# Get the AWS region name
region = boto_session.region_name

# Get the default S3 bucket name for SageMaker
bucket_name = sagemaker.Session().default_bucket()

# Define a prefix for the S3 bucket
bucket_prefix = "from-idea-to-prod/xgboost"

# Create a new SageMaker session
sm_session = sagemaker.Session()

# Create a new SageMaker client
sm_client = boto_session.client("sagemaker")

# Get the execution role for SageMaker
sm_role = sagemaker.get_execution_role()

# Define the local path to the dataset file
dataset_file_local_path = "data/bank-additional/bank-additional-full.csv"

# Set the initialized flag to True
initialized = True

# Print the SageMaker execution role
print(sm_role)

In [ ]:
# Store some variables to keep the value between the notebooks
%store bucket_name
%store bucket_prefix
%store sm_role
%store region
%store initialized
%store dataset_file_local_path

In [ ]:
NOTEBOOK_METADATA_FILE = "/opt/ml/metadata/resource-metadata.json"
domain_id = None

# Check if the metadata file exists
if os.path.exists(NOTEBOOK_METADATA_FILE):
    # Read and parse the metadata file
    with open(NOTEBOOK_METADATA_FILE, "rb") as f:
        metadata = json.loads(f.read())
        domain_id = metadata.get('DomainId')
        space_name = metadata.get('SpaceName')
        print(f"SageMaker domain id: {domain_id}")

# Raise an exception if the space name is not found
if not space_name:
    raise Exception(f"Cannot find the current space name. Make sure you run this notebook in a JupyterLab in the SageMaker Studio")
else:
    print(f"Space name: {space_name}")

# Describe the SageMaker space
r = sm_client.describe_space(DomainId=domain_id, SpaceName=space_name)
user_profile_name = r['OwnershipSettings']['OwnerUserProfileName']

# Ensure the user profile name is not empty
assert(user_profile_name)
print(f"User profile: {user_profile_name}")

# Store variables for later use
%store domain_id
%store space_name
%store user_profile_name

## Setting up MLflow Tracking Server

In [ ]:
# Find an active MLflow server in the account
r = boto3.client("sagemaker").list_mlflow_tracking_servers(
    TrackingServerStatus='Created',
)['TrackingServerSummaries']

if len(r) < 1:
    print("You don't have any running MLflow servers. Trying to find a server in the status 'Creating'...")

    r = boto3.client("sagemaker").list_mlflow_tracking_servers(
        TrackingServerStatus='Creating',
    )['TrackingServerSummaries']

    if len(r) < 1:
        print("You don't have any MLflow server in the status 'Creating'. Run the next code cell to create a new one.")
        mlflow_arn = None
        mlflow_name = None
    else:
        mlflow_arn = r[0]['TrackingServerArn']
        mlflow_name = r[0]['TrackingServerName']
        print(f"You have an MLflow server {mlflow_arn} in the status 'Creating', going to use this one")
else:
    mlflow_arn = r[0]['TrackingServerArn']
    mlflow_name = r[0]['TrackingServerName']
    print(f"You have {len(r)} running MLflow server(s). Get the first server ARN:{mlflow_arn}")

In [ ]:
# This code cell creates a new MLflow server
if not mlflow_arn:
    ts = strftime('%d-%H-%M-%S', gmtime())
    mlflow_name = f"mlflow-{domain_id}-{ts}"
    r = boto3.client("sagemaker").create_mlflow_tracking_server(
        TrackingServerName=mlflow_name,
        ArtifactStoreUri=f"s3://{bucket_name}/mlflow/{ts}",
        RoleArn=sm_role,
        AutomaticModelRegistration=True,
    )

    mlflow_arn = r['TrackingServerArn']
    print(f"Server creation request succeded. The server {mlflow_arn} is being created.")


In [ ]:
print(f'MLflow server name: {mlflow_name}')
print(f'MLflow server ARN: {mlflow_arn}')

## Install Docker

In [ ]:
# check that docker enabled in the SageMaker domain
docker_settings = sm_client.describe_domain(DomainId=domain_id)['DomainSettings'].get('DockerSettings')
docker_enabled = False

if docker_settings:
    if docker_settings.get('EnableDockerAccess') in ['ENABLED']:
        print(f"The docker access is ENABLED in the domain {domain_id}")
        docker_enabled = True

if not docker_enabled:
    raise Exception(f"You must enable docker access in the domain to use Studio local mode")

If the above code cell raised an exception that the docker access is not enabled, you need to enable the access. Enable dockaer access using CLI

`aws sagemaker update-domain --domain-id <DOMAIN-ID> --domain-settings-for-update DockerSettings={EnableDockerAccess='ENABLED'}`

In [ ]:
print(f"Domain id: {domain_id}")

In [ ]:
%%bash

# see https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository
sudo apt-get update
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

## Currently only Docker version 20.10.X is supported in Studio: see https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html
# pick the latest patch from:
# apt-cache madison docker-ce | awk '{ print $3 }' | grep -i 20.10
VERSION_STRING=5:20.10.24~3-0~ubuntu-jammy
sudo apt-get install docker-ce-cli=$VERSION_STRING docker-compose-plugin -y

# validate the Docker Client is able to access Docker Server at [unix:///docker/proxy.sock]
docker version

## Load the data

In [ ]:
!wget -P data/ -N https://archive.ics.uci.edu/static/public/222/bank+marketing.zip --no-check-certificate ## Unzip the dataset

In [ ]:
with zipfile.ZipFile("data/bank+marketing.zip", "r") as z:
    print("Unzipping bank+marketing...")
    z.extractall("data")

with zipfile.ZipFile("data/bank-additional.zip", "r") as z:
    print("Unzipping bank-additional...")
    z.extractall("data")

print("Done")

## Upload the data to S3
input_s3_url = sagemaker.Session().upload_data(
    path=dataset_file_local_path,
    bucket=bucket_name,
    key_prefix=f"{bucket_prefix}/input"
)
print(f"Upload the dataset to {input_s3_url}")

%store input_s3_url